# 25 — B1 EasyOCR producer (GPU, resumable B1 shard)

Notebook này chỉ xử lý **5.301 frame B1** đã thắng screening notebook 24. Quang
và Trọng chạy hai shard disjoint; marker theo video cho phép rerun bỏ qua phần đã
xong. Nó đọc deterministic JPEG tar của notebook 13 và ghi research artifact,
không sửa BTC/live index, không đọc held-out.

Chạy `T4/P100`, bật Internet và GitHub/Modal secrets. Quang giữ
`RUNNER="quang"`; Trọng đổi cả `RUNNER` và `CONFIRM_SHARD` thành `"trong"`.
Hai notebook 25 và 26 chạy song song ở hai runtime riêng; không chạy chung kernel.

## 0 — Chỉ sửa RUNNER/CONFIRM_SHARD

In [ ]:
from pathlib import Path

RUNNER = "quang"
RUNNERS = ("quang", "trong")
RUN_SCOPE = "SHARD"
CONFIRM_SHARD = "quang"
PIPELINE = "easyocr"
STAGES = ('n5_ocr_easyocr',)
SESSION_MAX_MINUTES = 600

EXPECTED_NB24_COMPLETE_SHA256 = "39950f697a1bfa9d6e6c9b78e220b8b62da48a99d8465b6f9b46a9c6b16a5c10"
EXPECTED_PARITY_FREEZE_SHA256 = "3b9993755bb19000e80a9c7bd9ccdde1e9a4648dd1950167d0759f9a14227180"
EXPECTED_B1_COUNT = 5301
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/b1-easyocr-producer")
REMOTE, OUTPUT = WORK / "remote", WORK / "output"
for path in (WORK, REMOTE, OUTPUT): path.mkdir(parents=True, exist_ok=True)
if RUNNER not in RUNNERS or RUN_SCOPE != "SHARD" or CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD lock is closed or runner is invalid")

## 1 — Repo, environment và Modal

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from collections import defaultdict
from datetime import datetime, timezone

def run(command, *, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode: raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1800:]}")
    return result.stdout.strip()

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient(); token = secrets.get_secret("GITHUB_TOKEN")
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
askpass = WORK / "git-askpass.sh"
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8"); askpass.chmod(0o700)
env = os.environ.copy(); env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO): raise RuntimeError("Restart Session: Kaggle checkout has local changes")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=env); run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else: run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=env)
finally: askpass.unlink(missing_ok=True)
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-btc-images.txt")])

sys.path.insert(0, str(REPO / "src"))

import torch
import yaml
from mfe.contracts.ids import build_keyframe_id
from mfe.offline.bundle import BundleLayout
from mfe.offline.keyframe_augmentation.common import atomic_write_json, read_jsonl, sha256_file, sha256_json
from mfe.offline.stages import Keyframe, StageContext, stage_n5_ocr_easyocr, stage_n5_ocr_paddle, stage_n5_ocr_vietocr, stage_n6_caption_qwen, stage_n6_ocr_qwen25, unload_models
from mfe.offline.transport import ModalVolumeTransport
if not torch.cuda.is_available(): raise RuntimeError("GPU is required")
repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
cfg = yaml.safe_load((REPO / "config/keyframe_augmentation_kaggle.yaml").read_text(encoding="utf-8"))
research = ModalVolumeTransport(cfg["output"]["volume"], create_if_missing=False)
root = cfg["output"]["root"].rstrip("/")
nb23_root = f"{root}/gate1-feature-parity-worklist-v1"; nb24_root = f"{root}/gate1-siglip-b0-b4-screen-v1"; output_root = f"{root}/gate1-b1-easyocr-v1"
print({"repo_commit": repo_commit, "gpu": torch.cuda.get_device_name(0), "runner": RUNNER, "pipeline": PIPELINE, "stages": STAGES})

## 2 — Khóa B1 và chia hai runner

In [ ]:
nb24_path, freeze_path, worklist_path = REMOTE / "nb24-COMPLETE.json", REMOTE / "parity-freeze.json", REMOTE / "parity-worklist.jsonl"
research.get(f"{nb24_root}/COMPLETE.json", nb24_path); research.get(f"{nb23_root}/parity-freeze.json", freeze_path); research.get(f"{nb23_root}/feature-parity-worklist.jsonl", worklist_path)
if sha256_file(nb24_path) != EXPECTED_NB24_COMPLETE_SHA256 or sha256_file(freeze_path) != EXPECTED_PARITY_FREEZE_SHA256: raise RuntimeError("Pinned lineage SHA drift")
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
if sha256_file(worklist_path) != freeze["files"]["feature-parity-worklist.jsonl"]: raise RuntimeError("Parity worklist SHA drift")
all_rows = read_jsonl(worklist_path)
b1_rows = [row for row in all_rows if "B1" in row["arms"]]
if len(b1_rows) != EXPECTED_B1_COUNT or len({row["keyframe_id"] for row in b1_rows}) != EXPECTED_B1_COUNT: raise RuntimeError("B1 identity/count drift")
by_video = defaultdict(list)
for row in b1_rows: by_video[row["video_id"]].append(row)
# Deterministic greedy balance by B1 frame count; same split in every producer.
loads = {runner: 0 for runner in RUNNERS}; owner = {}
for video_id, rows in sorted(by_video.items(), key=lambda item: (-len(item[1]), item[0])):
    runner = min(RUNNERS, key=lambda value: (loads[value], value)); owner[video_id] = runner; loads[runner] += len(rows)
assigned = {video_id: sorted(rows, key=lambda row: int(row["frame_idx"])) for video_id, rows in by_video.items() if owner[video_id] == RUNNER}
print({"b1_frames": len(b1_rows), "runner_loads": loads, "assigned_videos": len(assigned), "assigned_frames": sum(map(len, assigned.values()))})

## 3 — Writer, source integrity và resume

In [ ]:
class ResearchWriter:
    def __init__(self, video_id: str): self.video_id, self.artifacts = video_id, {}
    def write_text_part(self, artifact, video_id, mapping, *, model, params=None, facts=None, elapsed_s=None):
        cleaned = {str(key): str(value).strip() for key, value in mapping.items() if str(value).strip()}
        local = OUTPUT / "parts" / artifact / f"{video_id}.json"; local.parent.mkdir(parents=True, exist_ok=True)
        local.write_text(json.dumps(cleaned, ensure_ascii=False, sort_keys=True), encoding="utf-8")
        remote = f"{output_root}/parts/{artifact}/{video_id}.json"; digest = sha256_file(local); research.put(local, remote)
        if research.sha256(remote) != digest: raise RuntimeError(f"{video_id}/{artifact}: upload hash mismatch")
        result = {"artifact": artifact, "remote": remote, "sha256": digest, "rows": len(cleaned), "model": model, "params": params or {}, "facts": facts or {}, "elapsed_s": elapsed_s}
        self.artifacts[artifact] = result; return result

def marker_remote(video_id): return f"{output_root}/markers/{RUNNER}/{video_id}.json"
def completed(video_id):
    local = REMOTE / "markers" / RUNNER / f"{video_id}.json"
    try:
        research.get(marker_remote(video_id), local); marker = json.loads(local.read_text(encoding="utf-8")); unsigned = dict(marker); claimed = unsigned.pop("marker_payload_hash", None)
        if claimed != sha256_json(unsigned) or marker.get("status") != "COMPLETE" or marker.get("pipeline") != PIPELINE: return None
        for artifact in marker.get("artifacts", {}).values():
            if research.sha256(artifact["remote"]) != artifact["sha256"]: return None
        return marker
    except Exception: return None

def prepare_video(video_id, rows):
    local = WORK / "video" / video_id; shutil.rmtree(local, ignore_errors=True); images = local / "images"; images.mkdir(parents=True)
    source_marker = REMOTE / "source-markers" / f"{video_id}.json"; research.get(rows[0]["source_marker"], source_marker)
    marker = json.loads(source_marker.read_text(encoding="utf-8")); unsigned = dict(marker); claimed = unsigned.pop("marker_payload_hash", None)
    if claimed != rows[0]["source_marker_payload_hash"] or claimed != sha256_json(unsigned): raise RuntimeError(f"{video_id}: source marker invalid")
    tar_path = local / "images.tar"; research.get(rows[0]["source_images_tar"], tar_path)
    if sha256_file(tar_path) != marker["images_sha256"]: raise RuntimeError(f"{video_id}: source image tar SHA mismatch")
    with tarfile.open(tar_path) as archive: archive.extractall(images, filter="data")
    keyframes = []
    for n, row in enumerate(rows, 1):
        image = images / f"{int(row['frame_idx']):09d}.jpg"
        if not image.is_file(): raise RuntimeError(f"{video_id}: missing JPEG {image.name}")
        keyframes.append(Keyframe(n=n, pts_time=float(row["pts_time"]), fps=float(row["fps"]), frame_idx=int(row["frame_idx"]), image=image))
    return local, keyframes

## 4 — Chạy shard và chốt manifest

In [ ]:
STAGE_FUNCTIONS = {"n5_ocr_easyocr": stage_n5_ocr_easyocr, "n5_ocr_paddle": stage_n5_ocr_paddle, "n5_ocr_vietocr": stage_n5_ocr_vietocr, "n6_caption_qwen": stage_n6_caption_qwen, "n6_ocr_qwen25": stage_n6_ocr_qwen25}
pending = [video_id for video_id in sorted(assigned) if not completed(video_id)]
print({"verified_before": len(assigned) - len(pending), "pending": len(pending), "first_pending": pending[:3]})
started = time.perf_counter(); failures = []
for ordinal, video_id in enumerate(pending, 1):
    if ordinal > 1 and time.perf_counter() - started >= SESSION_MAX_MINUTES * 60: break
    local = None
    try:
        print(f"[{ordinal}/{len(pending)}] {video_id}: fetch B1 JPEG + {PIPELINE}", flush=True)
        local, keyframes = prepare_video(video_id, assigned[video_id]); writer = ResearchWriter(video_id)
        params = {"tensor_parallel_size": 2, "batch_size": 16, "prompt_variant": "aic", "text_first": True} if PIPELINE == "qwen25" else ({"device": "gpu", "lang": "vi", "vietocr_device": "cuda:0", "progress_every": 25} if PIPELINE == "paddle_vietocr" else {})
        ctx = StageContext(video_id, Path("/dev/null"), keyframes, writer, BundleLayout("research-b1"), local, params=params)
        for stage in STAGES: STAGE_FUNCTIONS[stage](ctx)
        marker = {"schema_version": 1, "status": "COMPLETE", "pipeline": PIPELINE, "runner": RUNNER, "video_id": video_id, "frame_count": len(keyframes), "keyframe_ids_sha256": sha256_json([build_keyframe_id(video_id, row["frame_idx"]) for row in assigned[video_id]]), "nb24_complete_sha256": EXPECTED_NB24_COMPLETE_SHA256, "artifacts": writer.artifacts, "repo_commit": repo_commit, "completed_at_utc": datetime.now(timezone.utc).isoformat()}
        marker["marker_payload_hash"] = sha256_json(marker); marker_path = OUTPUT / "markers" / RUNNER / f"{video_id}.json"; atomic_write_json(marker_path, marker); research.put(marker_path, marker_remote(video_id))
        print({"complete": video_id, "frames": len(keyframes)}, flush=True)
    except Exception as error:
        failures.append({"video_id": video_id, "error": str(error), "type": type(error).__name__}); print({"failed": failures[-1]}, flush=True)
    finally:
        if local: shutil.rmtree(local, ignore_errors=True)

verified = [video_id for video_id in sorted(assigned) if completed(video_id)]
manifest = {"schema_version": 1, "status": "COMPLETE" if len(verified) == len(assigned) else "PARTIAL", "pipeline": PIPELINE, "runner": RUNNER, "stages": list(STAGES), "assigned_videos": len(assigned), "assigned_frames": sum(map(len, assigned.values())), "verified_videos": len(verified), "verified_frames": sum(len(assigned[value]) for value in verified), "failures_this_session": failures, "nb24_complete_sha256": EXPECTED_NB24_COMPLETE_SHA256, "created_at_utc": datetime.now(timezone.utc).isoformat()}
manifest["manifest_payload_hash"] = sha256_json(manifest); manifest_path = OUTPUT / f"runner-manifest-{RUNNER}.json"; atomic_write_json(manifest_path, manifest); research.put(manifest_path, f"{output_root}/runner-manifests/{RUNNER}.json")
unload_models(); print({"B1_FEATURE_PRODUCER": PIPELINE, "runner": RUNNER, "status": manifest["status"], "verified_videos": len(verified), "assigned_videos": len(assigned), "verified_frames": manifest["verified_frames"], "assigned_frames": manifest["assigned_frames"], "manifest_sha256": sha256_file(manifest_path)})